In [ ]:
from pathlib import Path
from tifffile import imread
import json
import nd2
import numpy as np
import xarray
import re
import functools

from calmutils.imageio import save_tiff_imagej


def get_objective_name_from_nd2(fd):
    # if already opened, use as-is (don't close)
    if isinstance(fd, nd2.ND2File):
        meta = fd.metadata
    # otherwise open on-the-fly
    else:
        with nd2.ND2File(fd) as reader:
            meta = reader.metadata
    # get objective Name for first channel
    # NOTE: we assume the same objective is used throughout experiment
    return meta.channels[0].microscope.objectiveName


def load_correction_images(correction_info_path, obj, wls):

    with open(correction_info_path) as fd:
        correction_info = json.load(fd)

    for correction_info_i in correction_info:

        # skip non-matching objective
        if correction_info_i['objective'] != obj:
            continue

        darkfields = []
        flatfields = []

        for wl in wls:
            # error on non-existing images
            if wl not in correction_info_i['darkfields'] or wl not in correction_info_i['flatfields']:
                raise ValueError(f'No correction image for wavelength {wl} for objective {obj} found.')

            darkfield_i = imread(Path(correction_info_path).parent / correction_info_i['darkfields'][wl])

            # load flatfield, normalize to mean=1
            flatfield_i = imread(Path(correction_info_path).parent / correction_info_i['flatfields'][wl])
            flatfield_i /= flatfield_i.mean()

            darkfields.append(darkfield_i)
            flatfields.append(flatfield_i)

        # stack along C, to xarray
        darkfields = np.stack(darkfields)
        flatfields = np.stack(flatfields)
        darkfields = xarray.DataArray(darkfields, dims=['C', 'Y', 'X'])
        flatfields = xarray.DataArray(flatfields, dims=['C', 'Y', 'X'])
        # TODO: squeeze to handle single-channel?

        return flatfields, darkfields

    # error on no matching objective
    raise ValueError(f'No illumination correction for objective {obj} found.')


def get_wavelengths(reader):

    # if reader is not an open ND2File, try to open and call recursively
    if not isinstance(reader, nd2.ND2File):
        with nd2.ND2File(reader) as reader_i:
            return get_wavelengths(reader_i)

    # extraction of wavelength from text_info:
    pattern = r"(?:^|\n) +Line:\d;.*?(?=\n *Line:\d;|\n *(?=\S)|$)"
    matches = re.findall(pattern, reader.text_info['description'], re.DOTALL | re.MULTILINE)
    wavelengths = []
    for match in matches:
        if 'On' in match:
            wavelength = re.search(r'ExW:(\d+);', match).group(1)
            wavelengths.append(wavelength)
    return wavelengths


def xarray_to_tiffs(
    img,
    out_directory,
    out_prefix,
    split_dimensions=("C", "T", "P"),
    prefixes={"C": "_ch", "T": "_tp", "P": "_pos", "Z": "_z", "X": "_x", "Y": "_y"},
    use_indices=True,
    min_index_len=1,
    pixel_size = None,
):

    # TODO: similar to resave notebook, move to calmutils, use there as well?

    # if no pixel sizes are given, guess from xarray coordinate ticks (difference between adjacent)
    if pixel_size is None:
        psz_x = (img.coords['X'].values[1:] - img.coords['X'].values[:-1]).mean() if 'X' in img.coords else 1
        psz_y = (img.coords['Y'].values[1:] - img.coords['Y'].values[:-1]).mean() if 'Y' in img.coords else 1
        psz_z = (img.coords['Z'].values[1:] - img.coords['Z'].values[:-1]).mean() if 'Z' in img.coords else 1
        pixel_size = [psz_z, psz_y, psz_x]

    # find which of the selected split dimensions are present
    present_split_dimensions = [d for d in split_dimensions if d in img.dims]

    # handle no splitting
    if len(present_split_dimensions) == 0:

        # construct filename, dimension names
        out_filename = out_prefix + '.tif'
        out_filename = Path(out_directory) / out_filename
        axes = "".join([d for d in img.dims])

        save_tiff_imagej(
            out_filename,
            img.values.squeeze(),
            axes=axes,
            distance_unit="micron",
            pixel_size=pixel_size
        )

        return

    # group by split dimensions
    for idx, sub_img in img.groupby(present_split_dimensions):

        # treat even single split dimension as list of one
        if len(present_split_dimensions) == 1:
            idx = [idx]

        # get integer indices if desired
        if use_indices:
            filename_idx = [img.get_index(d).get_loc(i) for d,i in zip(present_split_dimensions, idx)]
            filename_idx = [str(i).rjust(min_index_len, '0') for i in filename_idx]
        else:
            filename_idx = idx

        # construct out filename
        out_filename = out_prefix + "".join(prefixes[d] + i for d, i in zip(present_split_dimensions, filename_idx)) + '.tif'
        out_filename = Path(out_directory) / out_filename

        # save as tiff
        axes = "".join([d for d in img.dims if d not in split_dimensions])
        save_tiff_imagej(out_filename, sub_img.values.squeeze(), axes=axes, distance_unit="micron", pixel_size=pixel_size)

In [ ]:
in_path = '/Volumes/agl_data/Machine_Learning/Cell_cycle_classification/260311 weihua_wen'
in_subdirectory = ''

out_subdirectory = 'tif_illumination_corrected'

correction_info_path = '/Volumes/agl_data/Optimisation-Maintainance/FlatfieldCorrection/illumination_correction_spinning_disk.json'

In [ ]:
in_files = sorted((Path(in_path) / in_subdirectory).glob('*.nd2'))
in_files

In [ ]:
@functools.lru_cache(1)
def load_correction_images_cached(correction_info_path, obj, wls):
    return load_correction_images(correction_info_path, obj, wls)

out_dir = Path(in_path) / out_subdirectory
if not out_dir.exists():
    out_dir.mkdir()

for in_file in in_files:

    objective = get_objective_name_from_nd2(in_file)
    wavelengths = tuple(get_wavelengths(in_file))

    flatfields, darkfields = load_correction_images_cached(correction_info_path, objective, wavelengths)

    img = nd2.imread(in_file, xarray=True)

    img_corrected = (img - darkfields) / flatfields
    img_corrected = np.clip(img_corrected, np.iinfo(img.dtype).min, np.iinfo(img.dtype).max).astype(img.dtype)

    xarray_to_tiffs(img_corrected, out_dir, in_file.stem)

    print(f'processed {in_file}.')

## Testing code below

In [ ]:

with open(correction_info_path) as fd:
        correction_info = json.load(fd)

correction_info

In [ ]:
import matplotlib.pyplot as plt

for correction_info_i in correction_info:

    fig, axs = plt.subplots(ncols=4, figsize=(12, 4))
    fig.suptitle(correction_info_i['objective'])

    for ax, (wavelength, correction_image_path) in zip(axs, correction_info_i['flatfields'].items()):
        
        img = imread(Path(correction_info_path).parent / correction_image_path)
        ax.imshow(img, cmap='gray')
        ax.set_title(wavelength)

In [ ]:

get_objective_name_from_nd2('/Users/david/Desktop/20240903-c2c12-DAPI-40x001.nd2')


In [ ]:
obj = 'Plan Apo λ 100x Oil'
wls = ['405', '488']

In [ ]:
import xarray
import numpy as np

rng = np.random.default_rng(seed=42)
a = rng.integers(1,10,(4,4))

b = xarray.DataArray(np.arange(4) * 1, dims='x')
a = xarray.DataArray(a, dims=('y', 'x'))

(a * 0.6).astype(a.dtype)
a.groupby(['x'])

np.finfo(np.float64)
np.dtypes

In [ ]:

get_wavelengths('/Users/david/Desktop/20240823-mESC-DAPI-001.nd2')

# __name__
# img

In [ ]:
img = nd2.imread('/Users/david/Desktop/20240823-mESC-DAPI-001.nd2', xarray=True)

xarray_to_tiffs(img, '/Users/david/Desktop/test1', 'img')